# Producer C — Camera Event Stream (Camera 3)

This notebook implements the Kafka producer for **Camera 3** of the AWAS traffic monitoring pipeline. It reads `camera_event_C.csv`, groups records into batches by `batch_id`, and publishes each batch to the Kafka topic `camera-events-C` at a controlled rate to simulate a live roadside camera stream.

---

## Role in the Pipeline

Producer C simulates an independent roadside camera feed by continuously publishing Camera 3 events to Kafka. Camera 3 is the final checkpoint in the monitored segment (speed limit: 90 km/h) and is the ending camera for the B→C average speed segment.

camera_event_C.csv → [Producer C] → Kafka topic: camera-events-C → Spark Structured Streaming

---

## EDA Findings

| Check | Finding |
|---|---|
| Camera IDs | Camera 3 only |
| Timestamp ordering | Not ordered within batches |
| Null speed readings | None found |
| Duplicate `car_plate` | 147 duplicates in vehicle.csv |
| `batch_id` | Local to Producer C (1–414,871) |
| Speed limit | 90 km/h — strictest of the three cameras |

These findings justified using Spark watermarks and time-window joins rather than batch_id matching. Camera 3's lower speed limit (90 km/h vs 110 km/h at cameras 1 and 2) means it is the most likely source of average-speed violations on the B→C segment.

---

## Event Payload

Each Kafka message contains:

- `event_id` — unique camera event identifier
- `batch_id` — batch grouping key (local to Producer C)
- `car_plate` — vehicle registration plate (whitespace-stripped)
- `camera_id` — always `3` for this producer
- `timestamp` — original event time when the vehicle passed Camera 3
- `speed_reading` — recorded speed in km/h, rounded to 2 dp
- `producer_id` — always `'C'`
- `produced_at` — UTC ingestion timestamp added at publish time
- `batch_size` — total events in this batch for consumer completeness checks

`timestamp` stores the original event time; `produced_at` stores Kafka ingestion time. Spark uses `timestamp` for watermarking and joins.

---

## Key Parameters

| Parameter | Value | Purpose |
|---|---|---|
| `BATCH_SLEEP_SECS` | `2` | Simulates streaming cadence |
| `TOPIC` | `camera-events-C` | Dedicated Kafka topic for Camera 3 |
| `api_version` | `(0, 10)` | Matches fit3182/kafka broker version |
| Error handling | `try/except` | Skips malformed rows without crashing |

---

## batch_id Behaviour

`batch_id` groups events into publishing batches. Producer C has **414,871 unique batch IDs** — the most of any producer — with most batches containing only 1–2 events. This reflects Camera 3 capturing fewer simultaneous vehicles, consistent with traffic thinning toward the end of the monitored segment. Spark joins do not use batch_id; matching is done on `car_plate` within a time window.

In [1]:
# Use %pip to ensure the package is linked to this specific notebook kernel
%pip install kafka-python

Note: you may need to restart the kernel to use updated packages.


In [2]:
import csv
import json
import time
from datetime import datetime, timezone
from collections import defaultdict
from kafka import KafkaProducer          
from kafka.errors import KafkaError   

## Implementation Overview

The producer consists of three main functions:

1. **`load_batches(filepath)`** — reads the CSV, groups rows by `batch_id`, converts fields to correct types, strips whitespace from `car_plate`, and skips malformed rows safely with a warning log.

2. **`connect_producer(broker)`** — creates and connects a JSON-serialised `KafkaProducer` to the Kafka broker.

3. **`publish_batches(producer, batches)`** — publishes one batch at a time, stamps `produced_at` and `batch_size` on every event, flushes messages to ensure broker acknowledgement, logs the next checkpoint batch_id for restart purposes, and sleeps between batches to simulate streaming.

For demonstration purposes, the producer may be stopped after ~10–15 batches since full dataset completion is not required during live demo execution.

In [3]:
# Config
HOST_IP = '192.168.0.175'
KAFKA_BROKER = '192.168.0.175:9092'
TOPIC               = 'camera-events-C'
PRODUCER_ID         = 'C'
DATA_FILE           = '../data/camera_event_C.csv'
BATCH_SLEEP_SECS= 2    # seconds between batches

In [4]:
def load_batches(filepath):
    """
    Read camera event CSV and group rows by batch_id.

    Args:
        filepath (str): Path to the camera event CSV file.

    Returns:
        list: Sorted list of (batch_id, [events]) tuples ordered by batch_id.
    """
    batches = defaultdict(list)
    skipped = 0

    with open(filepath, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                event = {
                    'event_id'     : str(row['event_id']),
                    'batch_id'     : int(row['batch_id']),
                    'car_plate'    : row['car_plate'].strip(),
                    'camera_id'    : int(row['camera_id']),
                    'timestamp'    : row['timestamp'],
                    'speed_reading': round(float(row['speed_reading']), 2),
                    'producer_id'  : PRODUCER_ID
                }
                batches[event['batch_id']].append(event)
            except (KeyError, ValueError) as e:
                skipped += 1
                print(f'[WARN] Skipping malformed row: {e} | row={row}')

    total_events = sum(len(v) for v in batches.values())
    print(f'[INFO] Loaded {total_events} events across {len(batches)} batches. '
          f'Skipped {skipped} malformed rows.')
    return sorted(batches.items())

In [5]:
def connect_producer(broker):
    """
    Connect to the Kafka broker and return a producer instance.

    Args:
        broker (str): Kafka broker address in host:port format.

    Returns:
        KafkaProducer: Connected producer that serialises messages as JSON.
    """
    producer = KafkaProducer(
        bootstrap_servers=[broker],
        value_serializer=lambda v: json.dumps(v).encode('utf-8'),
        api_version=(2, 5, 0) 
    )
    print(f'[INFO] Producer {PRODUCER_ID} connected → {broker} | topic: {TOPIC}')
    return producer

In [6]:
def publish_batches(producer, batches):
    """
    Publish all batches to the Kafka topic one batch at a time.

    Per batch: stamps produced_at and batch_size on each event,
    sends all events, flushes, then sleeps before next batch.
    Logs checkpoint (next batch_id) so restarts are possible after interruption.

    Args:
        producer (KafkaProducer): Connected Kafka producer.
        batches (list): Sorted list of (batch_id, [events]) tuples.
    """
    total_sent = 0

    for batch_id, events in batches:
        produced_at = datetime.now(timezone.utc).isoformat()
        batch_size  = len(events)
        batch_sent  = 0

        for event in events:
            event['produced_at'] = produced_at
            event['batch_size']  = batch_size
            try:
                producer.send(TOPIC, value=event)
                batch_sent += 1
            except KafkaError as e:
                print(f'[ERROR] Failed to send event {event["event_id"]}: {e}')

        producer.flush()
        total_sent += batch_sent
        print(f'[batch_id={batch_id:>6}] sent={batch_sent:>3} | '
              f'total={total_sent:>6} | next checkpoint: batch_id={batch_id + 1}')
        time.sleep(BATCH_SLEEP_SECS)

    print(f'[INFO] Producer C complete. Total sent: {total_sent}')

## Execution Notes

- Producers A, B, and C should be run **concurrently in separate Jupyter tabs** during the demo so Spark receives events from all three topics simulttneously.
- If your network changes (e.g. reconnecting to hotspot), update `HOST_IP` before re-running — the broker address is tied to your machine's current IPv4.
- If a `NoBrokersAvailable` eror appears, confirm the Kafka container is running with `docker ps` and that the IP matches `ipconfig` output.
- Camera 3 enforces a **90 km/h limit** — the strictest in the pipeline. Any vehicle averaging above 90 km/h between Camera 2 and Camera 3 triggers an average-speed violation at this endpoint.

In [ ]:
#Main:
batches  = load_batches(DATA_FILE)
producer = connect_producer(KAFKA_BROKER)

try:
    publish_batches(producer, batches)
except KeyboardInterrupt:
    print('[INFO] Producer C interrupted by user.')
finally:
    producer.close()
    print('[INFO] Producer C connection closed.')

[INFO] Loaded 556340 events across 414871 batches. Skipped 0 malformed rows.
[INFO] Producer C connected → 192.168.0.175:9092 | topic: camera-events-C
[batch_id=     1] sent=  1 | total=     1 | next checkpoint: batch_id=2
[batch_id=     2] sent=  1 | total=     2 | next checkpoint: batch_id=3
[batch_id=     3] sent=  2 | total=     4 | next checkpoint: batch_id=4
[batch_id=     4] sent=  2 | total=     6 | next checkpoint: batch_id=5
[batch_id=     5] sent=  1 | total=     7 | next checkpoint: batch_id=6
[batch_id=     6] sent=  1 | total=     8 | next checkpoint: batch_id=7
[batch_id=     7] sent=  2 | total=    10 | next checkpoint: batch_id=8
[batch_id=     8] sent=  1 | total=    11 | next checkpoint: batch_id=9
[batch_id=     9] sent=  2 | total=    13 | next checkpoint: batch_id=10
[batch_id=    10] sent=  2 | total=    15 | next checkpoint: batch_id=11
[batch_id=    11] sent=  1 | total=    16 | next checkpoint: batch_id=12
[batch_id=    12] sent=  1 | total=    17 | next check

[batch_id=   112] sent=  1 | total=   146 | next checkpoint: batch_id=113
[batch_id=   113] sent=  1 | total=   147 | next checkpoint: batch_id=114
[batch_id=   114] sent=  2 | total=   149 | next checkpoint: batch_id=115
[batch_id=   115] sent=  1 | total=   150 | next checkpoint: batch_id=116
[batch_id=   116] sent=  2 | total=   152 | next checkpoint: batch_id=117
[batch_id=   117] sent=  1 | total=   153 | next checkpoint: batch_id=118
[batch_id=   118] sent=  1 | total=   154 | next checkpoint: batch_id=119
[batch_id=   119] sent=  1 | total=   155 | next checkpoint: batch_id=120
[batch_id=   120] sent=  2 | total=   157 | next checkpoint: batch_id=121
[batch_id=   121] sent=  1 | total=   158 | next checkpoint: batch_id=122
[batch_id=   122] sent=  1 | total=   159 | next checkpoint: batch_id=123
[batch_id=   123] sent=  1 | total=   160 | next checkpoint: batch_id=124
[batch_id=   124] sent=  1 | total=   161 | next checkpoint: batch_id=125
[batch_id=   125] sent=  2 | total=   

[batch_id=   223] sent=  1 | total=   290 | next checkpoint: batch_id=224
[batch_id=   224] sent=  2 | total=   292 | next checkpoint: batch_id=225
[batch_id=   225] sent=  1 | total=   293 | next checkpoint: batch_id=226
[batch_id=   226] sent=  1 | total=   294 | next checkpoint: batch_id=227
[batch_id=   227] sent=  1 | total=   295 | next checkpoint: batch_id=228
[batch_id=   228] sent=  2 | total=   297 | next checkpoint: batch_id=229
[batch_id=   229] sent=  2 | total=   299 | next checkpoint: batch_id=230
[batch_id=   230] sent=  2 | total=   301 | next checkpoint: batch_id=231
[batch_id=   231] sent=  1 | total=   302 | next checkpoint: batch_id=232
[batch_id=   232] sent=  1 | total=   303 | next checkpoint: batch_id=233
[batch_id=   233] sent=  1 | total=   304 | next checkpoint: batch_id=234
[batch_id=   234] sent=  2 | total=   306 | next checkpoint: batch_id=235
[batch_id=   235] sent=  3 | total=   309 | next checkpoint: batch_id=236
[batch_id=   236] sent=  3 | total=   

[batch_id=   334] sent=  2 | total=   449 | next checkpoint: batch_id=335
[batch_id=   335] sent=  1 | total=   450 | next checkpoint: batch_id=336
[batch_id=   336] sent=  1 | total=   451 | next checkpoint: batch_id=337
[batch_id=   337] sent=  1 | total=   452 | next checkpoint: batch_id=338
[batch_id=   338] sent=  1 | total=   453 | next checkpoint: batch_id=339
[batch_id=   339] sent=  1 | total=   454 | next checkpoint: batch_id=340
[batch_id=   340] sent=  1 | total=   455 | next checkpoint: batch_id=341
[batch_id=   341] sent=  1 | total=   456 | next checkpoint: batch_id=342
[batch_id=   342] sent=  1 | total=   457 | next checkpoint: batch_id=343
[batch_id=   343] sent=  2 | total=   459 | next checkpoint: batch_id=344
[batch_id=   344] sent=  1 | total=   460 | next checkpoint: batch_id=345
[batch_id=   345] sent=  2 | total=   462 | next checkpoint: batch_id=346
[batch_id=   346] sent=  2 | total=   464 | next checkpoint: batch_id=347
[batch_id=   347] sent=  1 | total=   

[batch_id=   445] sent=  1 | total=   597 | next checkpoint: batch_id=446
[batch_id=   446] sent=  1 | total=   598 | next checkpoint: batch_id=447
[batch_id=   447] sent=  1 | total=   599 | next checkpoint: batch_id=448
[batch_id=   448] sent=  2 | total=   601 | next checkpoint: batch_id=449
[batch_id=   449] sent=  1 | total=   602 | next checkpoint: batch_id=450
[batch_id=   450] sent=  2 | total=   604 | next checkpoint: batch_id=451
[batch_id=   451] sent=  1 | total=   605 | next checkpoint: batch_id=452
[batch_id=   452] sent=  1 | total=   606 | next checkpoint: batch_id=453
[batch_id=   453] sent=  1 | total=   607 | next checkpoint: batch_id=454
[batch_id=   454] sent=  1 | total=   608 | next checkpoint: batch_id=455
[batch_id=   455] sent=  1 | total=   609 | next checkpoint: batch_id=456
[batch_id=   456] sent=  2 | total=   611 | next checkpoint: batch_id=457
[batch_id=   457] sent=  1 | total=   612 | next checkpoint: batch_id=458
[batch_id=   458] sent=  1 | total=   

...

[batch_id=  3640] sent=  1 | total=  4910 | next checkpoint: batch_id=3641
[batch_id=  3641] sent=  2 | total=  4912 | next checkpoint: batch_id=3642
[batch_id=  3642] sent=  1 | total=  4913 | next checkpoint: batch_id=3643
[batch_id=  3643] sent=  1 | total=  4914 | next checkpoint: batch_id=3644
[batch_id=  3644] sent=  1 | total=  4915 | next checkpoint: batch_id=3645
[batch_id=  3645] sent=  1 | total=  4916 | next checkpoint: batch_id=3646
[batch_id=  3646] sent=  1 | total=  4917 | next checkpoint: batch_id=3647
[batch_id=  3647] sent=  1 | total=  4918 | next checkpoint: batch_id=3648
[batch_id=  3648] sent=  1 | total=  4919 | next checkpoint: batch_id=3649
[batch_id=  3649] sent=  1 | total=  4920 | next checkpoint: batch_id=3650
[batch_id=  3650] sent=  1 | total=  4921 | next checkpoint: batch_id=3651
[batch_id=  3651] sent=  1 | total=  4922 | next checkpoint: batch_id=3652
[batch_id=  3652] sent=  1 | total=  4923 | next checkpoint: batch_id=3653
[batch_id=  3653] sent=  

[batch_id=  3750] sent=  1 | total=  5054 | next checkpoint: batch_id=3751
[batch_id=  3751] sent=  1 | total=  5055 | next checkpoint: batch_id=3752
[batch_id=  3752] sent=  1 | total=  5056 | next checkpoint: batch_id=3753
[batch_id=  3753] sent=  1 | total=  5057 | next checkpoint: batch_id=3754
[batch_id=  3754] sent=  3 | total=  5060 | next checkpoint: batch_id=3755
[batch_id=  3755] sent=  2 | total=  5062 | next checkpoint: batch_id=3756
[batch_id=  3756] sent=  5 | total=  5067 | next checkpoint: batch_id=3757
[batch_id=  3757] sent=  1 | total=  5068 | next checkpoint: batch_id=3758
[batch_id=  3758] sent=  1 | total=  5069 | next checkpoint: batch_id=3759
[batch_id=  3759] sent=  1 | total=  5070 | next checkpoint: batch_id=3760
[batch_id=  3760] sent=  2 | total=  5072 | next checkpoint: batch_id=3761
[batch_id=  3761] sent=  1 | total=  5073 | next checkpoint: batch_id=3762
[batch_id=  3762] sent=  1 | total=  5074 | next checkpoint: batch_id=3763
[batch_id=  3763] sent=  

[batch_id=  3860] sent=  1 | total=  5214 | next checkpoint: batch_id=3861
[batch_id=  3861] sent=  1 | total=  5215 | next checkpoint: batch_id=3862
[batch_id=  3862] sent=  3 | total=  5218 | next checkpoint: batch_id=3863
[batch_id=  3863] sent=  1 | total=  5219 | next checkpoint: batch_id=3864
[batch_id=  3864] sent=  1 | total=  5220 | next checkpoint: batch_id=3865
[batch_id=  3865] sent=  1 | total=  5221 | next checkpoint: batch_id=3866
[batch_id=  3866] sent=  1 | total=  5222 | next checkpoint: batch_id=3867
[batch_id=  3867] sent=  1 | total=  5223 | next checkpoint: batch_id=3868
[batch_id=  3868] sent=  1 | total=  5224 | next checkpoint: batch_id=3869
[batch_id=  3869] sent=  1 | total=  5225 | next checkpoint: batch_id=3870
[batch_id=  3870] sent=  1 | total=  5226 | next checkpoint: batch_id=3871
[batch_id=  3871] sent=  1 | total=  5227 | next checkpoint: batch_id=3872
[batch_id=  3872] sent=  3 | total=  5230 | next checkpoint: batch_id=3873
[batch_id=  3873] sent=  

[batch_id=  3970] sent=  1 | total=  5358 | next checkpoint: batch_id=3971
[batch_id=  3971] sent=  2 | total=  5360 | next checkpoint: batch_id=3972
[batch_id=  3972] sent=  1 | total=  5361 | next checkpoint: batch_id=3973
[batch_id=  3973] sent=  1 | total=  5362 | next checkpoint: batch_id=3974
[batch_id=  3974] sent=  1 | total=  5363 | next checkpoint: batch_id=3975
[batch_id=  3975] sent=  1 | total=  5364 | next checkpoint: batch_id=3976
[batch_id=  3976] sent=  1 | total=  5365 | next checkpoint: batch_id=3977
[batch_id=  3977] sent=  1 | total=  5366 | next checkpoint: batch_id=3978
[batch_id=  3978] sent=  1 | total=  5367 | next checkpoint: batch_id=3979
[batch_id=  3979] sent=  1 | total=  5368 | next checkpoint: batch_id=3980
[batch_id=  3980] sent=  1 | total=  5369 | next checkpoint: batch_id=3981
[batch_id=  3981] sent=  2 | total=  5371 | next checkpoint: batch_id=3982
[batch_id=  3982] sent=  1 | total=  5372 | next checkpoint: batch_id=3983
[batch_id=  3983] sent=  

[batch_id=  4080] sent=  1 | total=  5507 | next checkpoint: batch_id=4081
[batch_id=  4081] sent=  1 | total=  5508 | next checkpoint: batch_id=4082
[batch_id=  4082] sent=  1 | total=  5509 | next checkpoint: batch_id=4083
[batch_id=  4083] sent=  1 | total=  5510 | next checkpoint: batch_id=4084
[batch_id=  4084] sent=  2 | total=  5512 | next checkpoint: batch_id=4085
[batch_id=  4085] sent=  1 | total=  5513 | next checkpoint: batch_id=4086
[batch_id=  4086] sent=  1 | total=  5514 | next checkpoint: batch_id=4087
[batch_id=  4087] sent=  1 | total=  5515 | next checkpoint: batch_id=4088
[batch_id=  4088] sent=  1 | total=  5516 | next checkpoint: batch_id=4089
[batch_id=  4089] sent=  1 | total=  5517 | next checkpoint: batch_id=4090
[batch_id=  4090] sent=  2 | total=  5519 | next checkpoint: batch_id=4091
[batch_id=  4091] sent=  1 | total=  5520 | next checkpoint: batch_id=4092
[batch_id=  4092] sent=  1 | total=  5521 | next checkpoint: batch_id=4093
[batch_id=  4093] sent=  